# 08 — Counterfactual displacement spectrum (prereg A11)

**BLIND-SAFE up to section 7.** Sections 1–6 score only **random encoders** and the raw-pixel
reference, whose values A4 already makes disclosable. Section 7 scores trained checkpoints and is
gated on amendment **A11** being filed — the module refuses to load a trained checkpoint without
`--amendment`.

## What this measures, and why it is not another probe

Both datasets are complete factorial grids, so for any factor F, values v and v', and any
assignment z of the remaining factors, **both** images `g(z, F=v)` and `g(z, F=v')` exist. Exact
counterfactual pairs with context held fixed are available by construction, so the encoder's
response to changing F alone can be measured with **no probe of any kind**:

    Delta_F(z; v -> v') = W . (h(g(z, F=v')) - h(g(z, F=v)))

whitened by the probe-train covariance. Three statistics:

| statistic | meaning |
|---|---|
| `m_F`   | how far the encoder moves at all. At the null it does not move, so **no probe of any capacity** can recover F — a probe-free destruction certificate. |
| `rho_F` | fraction of displacement energy on one direction. A linear probe reads a factor off one direction, so this **bounds** linear readout quality. |
| `r_F`   | how many directions the factor's effect occupies. |

This is immune to the null saturation that sank the `G`-based headline: it computes no accuracy,
so it has no ceiling.

**Preregistered link (A11 d):** `(1 - rho_F)` predicts `Delta_G`. Confirm iff Spearman
`rho_s >= 0.5` with a bootstrap CI excluding 0, in its **own** Holm family. A failure is reported
as a failure and the destruction certificate stands alone.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Datasets + image cache
`--build-cache` decompresses once into an uncompressed memmap the loaders mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.dsprites --download --build-cache

In [ ]:
# Materialize every attached input into the repo layout, whatever shape it arrives in:
#   - a FLAT dataset (backbone_<cell>_strong_seed<N>.pt, calibration_<ds>.json)
#   - a NESTED prior-session output (results/** from Save Version) for resume
# Corrupt checkpoint archives are rejected rather than swept.
import json
import re
import shutil
import zipfile
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
INPUT = Path("/kaggle/input")
CELLS = ("color_strong", "control_strong", "position_strong")
KEEP = (".npz", ".json", ".jsonl", ".pt")


def intact(p):
    """A torch checkpoint is a zip; a CRC pass rejects truncated or corrupt copies."""
    if p.suffix != ".pt":
        return True
    try:
        with zipfile.ZipFile(p) as z:
            return z.testzip() is None
    except Exception:
        return False


def put(src, dst):
    if dst.exists() and intact(dst):
        return 0
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return 1


if not list(INPUT.glob("*")):
    raise SystemExit("no input datasets attached - Add Input the encoder + results dataset")

mirrored = backbones = calib = corrupt = 0

# 1. Nested prior-session outputs: mirror results/** verbatim (stacks, probe caches, meta).
# "*/**/results" also matches "*/results", so a dataset packed one or more folders
# deep (e.g. <ds>/probe_outputs/results/**) mirrors the same as a top-level one.
for root in sorted({p for p in INPUT.glob("*/**/results") if p.is_dir()}):
    for f in root.rglob("*"):
        if f.is_file() and f.suffix in KEEP:
            if not intact(f):
                print(f"SKIP corrupt: {f}")
                corrupt += 1
                continue
            mirrored += put(f, REPO / "results" / f.relative_to(root))

# 2. Encoder backbones, any layout, onto the one path section 9 globs.
for ck in sorted(INPUT.rglob("*.pt")):
    m = re.search(r"((?:color|control|position)_strong_seed\d+)", ck.as_posix())
    if not m or "last_ckpt" in ck.name:
        continue
    if not intact(ck):
        print(f"SKIP corrupt: {ck}")
        corrupt += 1
        continue
    backbones += put(ck, REPO / "results" / "encoders" / m.group(1) / "backbone.pt")

# 3. Calibration artifacts, any layout.
for j in sorted(INPUT.rglob("calibration_*.json")):
    calib += put(j, REPO / "results" / "calibration" / j.name)

print(f"mirrored {mirrored} prior result file(s) | {backbones} backbone(s) | "
      f"{calib} calibration file(s) | {corrupt} corrupt skipped\n")

for cell in CELLS:
    n = len(sorted((REPO / "results" / "encoders").glob(f"{cell}_seed*/backbone.pt")))
    print(f"{cell:18s} {n:>2d}/12 {'OK' if n == 12 else 'MISSING'}")
for ds in ("shapes3d", "dsprites"):
    p = REPO / "results" / "calibration" / f"calibration_{ds}.json"
    n = len(json.loads(p.read_text())["results"]) if p.exists() else 0
    print(f"calibration_{ds:9s} {n:>2d}/16 rows {'OK' if n == 16 else 'MISSING'}")

## 5. Preflight — the instrument, and the blind

Checks the A11 module is present, that every §5.4 requirement has a live implementation, and
that the blind guard actually refuses a trained checkpoint without `--amendment`. The last check
is the one that matters: a guard that does not fire is not a guard.

In [ ]:
import inspect
import numpy as np
from src.probes import displacement as D

checks = [
    ("req 1  monotonicity check",      hasattr(D, "monotonicity"), ""),
    ("req 2  fit/eval split",          "n_fit" in inspect.getsource(D.spectrum_stats), ""),
    ("req 2  bootstrap over contexts", "evl_by_ctx" in inspect.signature(D.bootstrap_stats).parameters, ""),
    ("req 3  epsilon_m null",          hasattr(D, "epsilon_m"), ""),
    ("req 4  probe-train whitening",   hasattr(D, "whitener"), ""),
    ("req 6  fixed context seed",      isinstance(getattr(D, "CONTEXT_SEED", None), int), ""),
]
for n, ok, _ in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n}")
assert all(ok for _, ok, _ in checks), "stale code — git pull, restart kernel"

# the blind guard must FIRE
import argparse
ns = argparse.Namespace(trained_encoders=["x/backbone.pt"], amendment=None, config=None,
                        dataset="shapes3d")
try:
    D.run(ns); raise AssertionError("BLIND GUARD DID NOT FIRE")
except SystemExit as e:
    assert "BLIND GUARD" in str(e), e
    print("\n  PASS  blind guard refuses a trained checkpoint without --amendment")

## 6. Random-encoder + pixel reference (BLIND-SAFE)

This is the null the destruction certificate is measured against, and it needs >= 2 random seeds
or `epsilon_m` is undefined. Roughly `n_contexts x n_values` forward passes per factor — cheap
next to a probe sweep, because nothing is trained.

In [ ]:
# A15 (b): the certificate REFUSES a verdict without a calibration, so this runs first.
# It scores a synthetic fixture of known Bayes decodability -- no encoder, no dataset,
# fully blind-safe -- and writes m_star + the unadjudicated band.
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --calibrate-threshold --threshold-file results/displacement/threshold_calibration.json

import json
from pathlib import Path
_c = json.loads(Path("results/displacement/threshold_calibration.json").read_text())
print("m_star =", _c["m_star"], "| unadjudicated band =", _c["unadjudicated_band"])
assert _c.get("m_star"), "calibration did not separate the fixture classes -- STOP"


In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset shapes3d --cell reference_shapes3d \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --n-contexts 512 --device cuda --num-workers 2 --out-root results/displacement

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset dsprites --cell reference_dsprites \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --n-contexts 512 --device cuda --num-workers 2 --out-root results/displacement

In [ ]:
import json
from pathlib import Path
import numpy as np

for cell in ("reference_shapes3d", "reference_dsprites"):
    p = Path(f"results/displacement/{cell}.json")
    if not p.exists():
        print(f"{cell}: NOT RUN"); continue
    r = json.loads(p.read_text())
    print(f"\n=== {cell} === contexts={r['n_contexts']} seed={r['context_seed']} "
          f"m_star={r.get('m_star')} var_fraction={r.get('var_fraction')}")
    for role, tags in r["roles"].items():
        first = next(iter(tags))
        print(f"  {role}:  (n={len(tags)}, retained rank "
              f"{tags[first]['whitening']['rank']}/{tags[first]['whitening']['d']})")
        for f in tags[first]["factors"]:
            st = [tags[t]["factors"][f] for t in tags]
            dp = np.mean([s["d_prime"] for s in st])
            sh = np.mean([s["shared_fraction"] for s in st])
            wc = np.mean([s["whitening_check"] for s in st])
            mono = np.mean([s["monotonicity"]["monotone_fraction"] for s in st])
            vd = {s["verdict"] for s in st}
            print(f"    {f:14s} d'={dp:8.4f}  shared={sh:.3f}  mono={mono:.2f}  "
                  f"wcheck={wc:.2f}  verdict={'/'.join(sorted(vd))}")
    print("  epsilon_m (REDUCTION test only, A15 a):",
          {k: round(v, 4) for k, v in r.get("epsilon_m", {}).items()})

## 7. Trained encoders — BLIND-GATED

**Do not run this cell until amendment A11 is filed in `preregistration/prereg.md`.** It is
filed as of 2026-08-13; the guard checks the file rather than trusting this sentence.

Running it produces trained-encoder targeted-factor values. That is exactly what A11 exists to
license, and exactly why the link test's effect-size target (`rho_s >= 0.5`, CI excluding 0) was
fixed in advance.

In [ ]:
from pathlib import Path
AMENDMENT = "A11-2026-08-13"
txt = Path("preregistration/prereg.md").read_text()
assert "### A11 " in txt and "(1 − ρ_F)" in txt.replace("(1 - rho_F)", "(1 − ρ_F)"), (
    "A11 is not in the frozen prereg. File it BEFORE computing rho_F on any trained encoder.")
print("A11 present — trained-encoder displacement is licensed.")
for cell, ds, pat in (("color_strong", "shapes3d", "results/encoders/color_strong_seed*/backbone.pt"),
                      ("control_strong", "shapes3d", "results/encoders/control_strong_seed*/backbone.pt"),
                      ("position_strong", "dsprites", "results/encoders/position_strong_seed*/backbone.pt")):
    n = len(sorted(Path().glob(pat)))
    print(f"  {cell:16s} {n:2d} checkpoint(s)")
    assert n == 12, (
        f"{cell}: '{pat}' resolved {n} checkpoints, expected 12. displacement.py only "
        "WARNS on too few encoders, so an unresolved glob would score zero trained "
        "encoders and write a result that looks valid.")

In [ ]:
# The pixel role is scored once in section 6; rescoring it here costs more than the
# encoders do and adds nothing.
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset shapes3d --cell color_strong --amendment A11-2026-08-13 \
    --trained-encoders results/encoders/color_strong_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --n-contexts 512 --no-pixel-reference --device cuda --num-workers 2 --out-root results/displacement

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset shapes3d --cell control_strong --amendment A11-2026-08-13 \
    --trained-encoders results/encoders/control_strong_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --n-contexts 512 --no-pixel-reference --device cuda --num-workers 2 --out-root results/displacement

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset dsprites --cell position_strong --amendment A11-2026-08-13 \
    --trained-encoders results/encoders/position_strong_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --n-contexts 512 --no-pixel-reference --device cuda --num-workers 2 --out-root results/displacement

## 8. The destruction certificate and the A11 (d) link test

**A15 (b) supersedes A11 (b).** The verdict is **eps-DESTROYED AT RESOLUTION `m_star`**
(`m_total < m_star`, `m_star` = 0.2888), licensing *"no probe separates F at
discriminability `m_star` on this encoder"* — **never** *"the information is absent"*.
A readout inside the unadjudicated band [0.0799, 1.0445] is reported as `unadjudicated`,
not as either verdict. `epsilon_m` is retained as a **reduction** test only (A15 a).

The link test runs per cell and within metric type (A13 a's unit, prereg FIX 1's
restriction). Where a group has fewer than 3 readouts it is reported as **not
evaluable** — a power limitation of the realized grid, never a refutation.

In [ ]:
import json
from pathlib import Path
import re

import numpy as np
from scipy.stats import spearmanr

CELLS = ("color_strong", "control_strong", "position_strong")

# Artifacts are read from the repo layout when notebook 06 wrote them here, and
# otherwise straight out of an attached dataset, at whatever depth it was packed
# (e.g. /kaggle/input/<ds>/probe_outputs/results/probes_pinned/<cell>/).
SEARCH_ROOTS = tuple([d / "results" for d in (Path.cwd(), *Path.cwd().parents)]
                    + [Path("/kaggle/working/probe-capacity-invariance/results"),
                       Path("/kaggle/input")])


def _find(rel):
    """First existing match for a results-relative path, across the search roots."""
    for base in SEARCH_ROOTS:
        if not base.is_dir():
            continue
        p = base / rel
        if p.exists():
            return p
        hits = sorted(base.glob(f"**/{rel}"))
        if hits:
            return hits[0]
    return None


def _seed_of(tag):
    m = re.search(r"seed(\d+)", tag)
    if m is None:
        raise ValueError(f"cannot read a seed index from displacement tag {tag!r}")
    return int(m.group(1))


def _by_seed(tags):
    return sorted(tags, key=_seed_of)


rows, m_stars, bands = [], set(), set()
for cell in CELLS:
    dp = _find(f"displacement/{cell}.json")
    hp = _find(f"probes_pinned/{cell}/hypothesis_report.json")
    mp = _find(f"probes_pinned/{cell}/meta.json")
    missing = [n for n, p in (("displacement", dp), ("hypothesis_report", hp),
                              ("meta", mp)) if p is None]
    if missing:
        print(f"{cell}: missing {', '.join(missing)}")
        continue
    print(f"{cell}: displacement <- {dp}\n{'':{len(cell) + 2}}probes      <- {hp.parent}")
    dj, hj = json.loads(dp.read_text()), json.loads(hp.read_text())
    tr, rnd = dj["roles"].get("trained", {}), dj["roles"].get("random", {})
    if not tr:
        print(f"{cell}: no trained role scored"); continue
    eps_m = dj.get("epsilon_m", {})
    m_stars.add(dj.get("m_star"))
    kind = {f["name"]: f["kind"] for f in json.loads(mp.read_text())["factors"]}
    dg = {r["factor"]: r for r in hj["h1"]["per_factor"]}
    sat = set(hj.get("null_saturation", {}).get("saturated_factors_flip", []))
    diag = set(hj.get("report", {}).get("diagnostic_only_factors", []))
    scored = next(iter(tr.values()))["factors"]
    for f in dg:
        if f not in scored:
            continue
        st = [tr[t]["factors"][f] for t in tr]
        rn = [rnd[t]["factors"][f] for t in rnd] if rnd else []
        e = eps_m.get(f, float("nan"))
        m_tr = float(np.mean([s["m"] for s in st]))
        m_rn = float(np.mean([s["m"] for s in rn])) if rn else float("nan")
        rows.append({
            "cell": cell, "factor": f,
            "d_prime": float(np.mean([s["d_prime"] for s in st])),
            "verdict": "/".join(sorted({s["verdict"] for s in st})),
            # A12 (b): rho_shared, the component one fixed readout sees. This is the
            # link test's predictor; rho (spectral concentration) is a diagnostic.
            "rho_shared": float(np.mean([s["shared_direction_fraction"] for s in st])),
            "rho_spectral": float(np.mean([s["rho"] for s in st])),
            "mono": float(np.mean([s["monotonicity"]["monotone_fraction"] for s in st])),
            "wcheck": float(np.mean([s["whitening_check"] for s in st])),
            # A15 (a): eps_m is a REDUCTION test, never a destruction verdict.
            "reduced": bool((m_rn - m_tr) > e) if e == e else None,
            "delta_g": dg[f]["delta_g"],
            "delta_g_per_seed": dg[f].get("delta_g_per_seed"),
            # NUMERIC seed order. sorted() on the tags is lexicographic
            # (seed0, seed1, seed10, seed11, seed2, ...) while delta_g_per_seed is
            # numeric (run_sweep.py sorts by _seed_from_path), so a positional zip of
            # the two mispairs every slot from index 2 on.
            "rho_shared_per_seed": [tr[t]["factors"][f]["shared_direction_fraction"]
                                    for t in _by_seed(tr)],
            "seeds": [_seed_of(t) for t in _by_seed(tr)],
            "kind": kind.get(f),
            "eligible": f not in sat and f not in diag,
        })

# The unadjudicated band is written by --calibrate-threshold, not by a scoring run.
calib = _find("displacement/threshold_calibration.json")
if calib is not None:
    cj = json.loads(calib.read_text())
    if cj.get("unadjudicated_band"):
        bands.add(tuple(cj["unadjudicated_band"]))
else:
    print("WARNING: displacement/threshold_calibration.json absent -- "
          "no unadjudicated band, verdicts unqualified.")

ms = m_stars.pop() if len(m_stars) == 1 else None
print(f"m_star = {ms}   unadjudicated band = {sorted(bands)[0] if bands else 'n/a'}")
print("verdict: d' < m_star -> eps-destroyed AT RESOLUTION m_star (A15 b). NOT 'absent'.\n")
print(f"{'cell':16s}{'factor':13s}{'d_prime':>9s}{'verdict':>15s}{'rho_shr':>9s}"
      f"{'mono':>6s}{'wchk':>6s}{'reduced':>8s}{'Delta_G':>9s}{'elig':>6s}")
for r in rows:
    print(f"{r['cell']:16s}{r['factor']:13s}{r['d_prime']:>9.4f}{r['verdict']:>15s}"
          f"{r['rho_shared']:>9.3f}{r['mono']:>6.2f}{r['wcheck']:>6.2f}"
          f"{str(r['reduced']):>8s}{r['delta_g']:>9.4f}{str(r['eligible']):>6s}")

band = sorted(bands)[0] if bands else None
if band:
    una = [r for r in rows if band[0] <= r["d_prime"] <= band[1]]
    if una:
        print(f"\nUNADJUDICATED ({len(una)}): "
              + ", ".join(f"{r['cell']}/{r['factor']}" for r in una)
              + f"  -- inside [{band[0]:.4f}, {band[1]:.4f}], reported as neither verdict.")

# --- A11 (d) / A12 (c) / A13 (a) link test -------------------------------------
# Computed under the FROZEN reading, which fixes both the unit and the grouping:
#   A13 (a): "one statistic per seed (12 PER CELL)"  -> per-cell, never pooled across cells
#   prereg FIX 1 (§3): "R^2 and normalized-accuracy are never pooled or RANK-COMPARED"
#                                                     -> within metric type only
# Spearman needs >= 3 points. Pooling across cells or across metric types would make
# the test evaluable where these rules say it is not, in the direction of the study,
# so neither is done. Non-evaluation is reported as non-evaluation, never as a null.
elig = [r for r in rows if r["eligible"]]
groups = {}
for r in elig:
    groups.setdefault((r["cell"], r["kind"]), []).append(r)

print(f"\nlink-test eligible readouts: {len(elig)}")
for (cell, k), grp in sorted(groups.items()):
    unit = "R^2" if k == "continuous" else "norm-acc"
    names = ", ".join(g["factor"] for g in grp)
    print(f"   {cell:16s} {unit:9s} n={len(grp)}  [{names}]"
          + ("" if len(grp) >= 3 else "   NOT EVALUABLE (Spearman needs n>=3)"))

def _boot_ci(vals, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    v = np.asarray(vals, float); v = v[np.isfinite(v)]
    if v.size < 2: return (float("nan"), float("nan"), float("nan"))
    d = np.array([np.mean(rng.choice(v, v.size, replace=True)) for _ in range(n)])
    lo, hi = np.nanpercentile(d, [2.5, 97.5])
    pv = 2.0 * min((d <= 0).mean(), (d >= 0).mean())
    return float(lo), float(hi), float(min(1.0, max(pv, 1.0 / n)))

evaluable = {g: v for g, v in groups.items() if len(v) >= 3}
if not evaluable:
    print("\nPRIMARY (A13 a) NOT EVALUABLE IN ANY CELL.")
    print("  Every (cell x metric-type) group has n<3 under A13 (a)'s per-cell unit and")
    print("  FIX 1's within-type restriction. The mechanism half of the contribution is")
    print("  NOT TESTABLE on the realized grid. Reported as a power limitation of the")
    print("  grid -- NOT as a refutation, and NOT as support. The A15 (b) certificate")
    print("  stands alone, exactly as A11 (d) pre-committed.")
else:
    res = []
    for (cell, k), grp in sorted(evaluable.items()):
        seeds = {tuple(g["seeds"]) for g in grp}
        assert len(seeds) == 1, f"{cell}/{k}: seed order disagrees across readouts: {seeds}"
        n_seed = min(len(g["delta_g_per_seed"]) for g in grp)
        rs = [spearmanr([1.0 - g["rho_shared_per_seed"][s] for g in grp],
                        [g["delta_g_per_seed"][s] for g in grp]).statistic
              for s in range(n_seed)]
        lo, hi, pv = _boot_ci(rs)
        res.append({"group": f"{cell}/{k}", "n": len(grp), "rs": float(np.nanmean(rs)),
                    "lo": lo, "hi": hi, "p": pv})
    # Holm WITHIN the link family (A11 d), never pooled with the H1-H4 families.
    for rank, r in enumerate(sorted(res, key=lambda r: r["p"])):
        r["p_holm"] = min(1.0, r["p"] * (len(res) - rank))
    print(f"\nPRIMARY (A13 a, per-seed, per-cell, within-type) -- Holm over {len(res)} test(s):")
    for r in sorted(res, key=lambda r: r["group"]):
        ok = (r["rs"] >= 0.5) and (r["lo"] > 0 or r["hi"] < 0) and r["p_holm"] < 0.05
        det = (not ok) and (r["lo"] > 0 or r["hi"] < 0) and r["p_holm"] < 0.05
        print(f"   {r['group']:26s} rho_s={r['rs']:+.3f} CI95[{r['lo']:+.3f},{r['hi']:+.3f}] "
              f"n={r['n']} p_holm={r['p_holm']:.3f} -> "
              + ("CONFIRMED" if ok else "DETECTED BUT IMMATERIAL" if det else "REFUTED"))
    print("  Target fixed in A11 (d): confirm iff rho_s >= 0.5 AND CI excludes 0.")

In [ ]:
# A15 (c) reduction controls. NEITHER is ground truth: greyscale leaks hue at
# L2 >= 61.4 and the value channel is ~16x tighter but still nonzero. A certificate
# that does not fire here is broken; firing is necessary, not sufficient.
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset shapes3d --cell reduction_grayscale --grayscale \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 --n-contexts 512 --no-pixel-reference --device cuda \
    --num-workers 2 --out-root results/displacement
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset shapes3d --cell reduction_value --value-channel \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 --n-contexts 512 --no-pixel-reference --device cuda \
    --num-workers 2 --out-root results/displacement


In [ ]:
# A15 (d): var_fraction in {0.95, 0.99, 0.999} is a PREREGISTERED sensitivity
# co-reported with every certificate verdict -- it bounds the verdict, not garnishes it.
for vf in (0.95, 0.999):
    !cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
        --dataset shapes3d --cell color_strong_vf{vf} --amendment A11-2026-08-13 \
        --trained-encoders results/encoders/color_strong_seed*/backbone.pt \
        --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 --var-fraction {vf} \
        --n-contexts 512 --no-pixel-reference --device cuda --num-workers 2 --out-root results/displacement


## 9. Persist for the next session
Click **Save Version**, then **Add Input -> this output** on the next run.

In [ ]:
import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files -> click 'Save Version'")